In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats
import pickle
import json
import copy

In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
rundirs = [
    'bomex_25m_r20251009',
    'bomex_25m_ehe18_r20251009',
    'bomex_25m_ehe22_r20251107',
    'bomex_25m_ehe21_r20251107',
]
casenames = [
    'CTL',
    'EHEall',
    'EHElower',
    'EHEupper',
]
casecolors = [ 
    'black',
    'green',
    'red',
    'blue',
]
stats = [] 
for rundir in rundirs:
    with open(f'{rundir}/pkl/csd_stats.pkl', 'rb') as f:
        stats.append(pickle.load(f))

In [ ]:
minmf = 0
maxmf = np.max([np.max(s[0]) for s in stats]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)

In [ ]:
nx, ny, nz, nt = 512, 512, 120, 241
dts = 0.5 # minute
dx = 25 # m
dy = 25 # m
dz = 25 # m
grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3
z = np.arange(dz/2, 3000., dz)
t = np.arange(0, 241)*dts # minutes
ti = np.arange(-0.5, 241., 1.)*dts
zi = np.arange(0., 3001., dz)

In [ ]:
%%script echo skipping

def total_wq(casename):
    # with open(f'{casename}/pkl/qclm.pkl', 'rb') as f:
    #     qclm = pickle.load(f)
    with open(f'{casename}/pkl/ws.pkl', 'rb') as f:
        ws = pickle.load(f)
    with open(f'{casename}/pkl/qts.pkl', 'rb') as f:
        qts = pickle.load(f)
    qfluxes = []
    qd = [
        ["MUa", "MDa", "DDa", "DUa", "ENa"],
        ["MUatl", "MDatl", "DDatl", "DUatl", "ENatl"],
        ["MUatul", "MDatul", "DDatul", "DUatul", "ENatul"],
        ["MUatula", "MDatula", "DDatula", "DUatula", 'ENatula'],
    ]
    for quads in qd:
        qflux = []
        for quad in quads:
            with open(f'{casename}/pkl/{quad}_qflux.pkl', 'rb') as f:
                qflux.append(pickle.load(f)*ws*qts)
        qfluxes.append(qflux)
    return np.asarray(qfluxes)
qfluxes = {}
qfluxes[ctl] = total_wq(ctl)
qfluxes[ehe18] = total_wq(ehe18)
total_wq = {}
total_wq[ctl] = qfluxes[ctl].sum(axis=1)
total_wq[ehe18] = qfluxes[ehe18].sum(axis=1)
total_wq_diff = total_wq[ehe18] - total_wq[ctl]
pop_labels = ['domain', 'tracked', 'tracked full', 'tracked full attached']
styles = ['--', ':', '-', 'None']
markers = [None, None, None, '+']

In [ ]:
def binned_mf(csd_stats, rundir, bins):
    global nx, ny, nz, nt
    print(nx, ny, nz, nt)
    with open(f'{rundir}/pkl/plume_all_mf.pkl', 'rb') as f:
        mf = pickle.load(f)
    clipped_mf = csd_stats[0] 
    attached_ind = csd_stats[-1]
    bin_ids = np.digitize(np.log10(clipped_mf), bins)
    nbins = len(bins) - 1
    # factor = 1.0/float(nx*ny*nt)
    factor = 1
    sum_mf = np.zeros((nbins, nz))
    mean_mf = np.zeros((nbins, nz))
    total_count = 0
    for bnm in range(nbins):
        bn = bnm + 1
        cn = attached_ind[bin_ids==bn]
        total_count += len(cn)
        print(bn, len(cn))
        if len(cn) > 0:
            sum_mf[bnm, :] = (mf[cn, :, :].sum(axis=1)*factor).sum(axis=0)
            mean_mf[bnm, :] = (mf[cn, :, :].sum(axis=1)*factor).mean(axis=0)
    return sum_mf, mean_mf

In [ ]:
nbins = 15 
bins = np.linspace(minmf, maxmf, nbins+1)
sum_mfs = []
mean_mfs = []
for s, d in zip(stats, rundirs):
    sum_mf, mean_mf = binned_mf(s, d, bins)
    sum_mfs.append(sum_mf)
    mean_mfs.append(mean_mf)

In [ ]:
def compare_mf(sum_mfs, mean_mfs, casenames, casecolors, minmf, maxmf, plot_top=2.0, nbins=9):

    global zi, dx, dy, nt
    bins = np.linspace(minmf, maxmf, nbins+1)

    # Create comparison plots for all EHE experiments
    fig, axs = plt.subplots(1, 3, figsize=(18, 12))
    
    for idx, s in enumerate(sum_mfs[1:]):
        ax = axs[idx]
        sum_mf_diff = (s - sum_mfs[0]) * dx * dy * 1.0e-3 / nt # tonnes/s
        print(f"{casenames[idx+1]}: max={sum_mf_diff.max()}, min={sum_mf_diff.min()}")
        
        # levels = np.concatenate([np.arange(-6, -0.1, 1), [-0.1], [-0.01, 0.01], [0.1], np.arange(1, 6.1, 1)])
        levels = np.concatenate([np.arange(-6, -0.1, 1), [-0.1, 0.1], np.arange(1, 6.1, 1)])*100.0
        # levels = np.concatenate([np.arange(-2, -0.1, 1), [-0.1, 0.1], np.arange(1, 6.1, 1)])
        cmap = copy.deepcopy(mpl.cm.bwr)
        norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
        cm = ax.pcolormesh(bins, zi*1.0e-3, sum_mf_diff.T, norm=norm, cmap=cmap, shading='flat')
        ax.axhline(0.6125, color='black', linestyle='--', linewidth=1.0)
        ax.set_ylim(0, plot_top)
        ax.set_xlabel(r'$\log_{10}\left<M_b\right>$ (kg/s)', fontsize=18)
        ax.set_title(f"{casenames[idx+1]} - CTL", fontsize=18)
        
        if idx == 0:
            ax.set_ylabel('Height (km)', fontsize=18)
        else:
            ax.set_yticklabels([])

        if idx == 2:        

            # Adjust spacing to prevent colorbar overlap
            
            fig.subplots_adjust(right=0.85)
            cbar_ax = fig.add_axes([0.87, axs[0].get_position().y0, 0.02, axs[0].get_position().height])
            cbar = plt.colorbar(cm, cax=cbar_ax)
            cbar.set_ticks(levels)
            cbar.ax.tick_params(labelsize=16)
            cbar.set_label(r"$\sum M$ diff (tonnes/s)", fontsize=18)
    
    # plt.tight_layout()
    plt.show()

    bins_to_plot = [10, 11, 12, 13 ,14]
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()

    # Adjust spacing to make panels closer together
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(mean_mfs[0][binno,:]*dx*dy*1.0e-3, z*1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(mean_mfs[1:]):
            ax.plot(m[binno,:]*dx*dy*1.0e-3, z*1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.set_xlim()
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        
        # Add labels only to left column and bottom row
        if iax == 0:
            ax.set_ylabel('Height (km)')

        # Simpler title with bin range
        ax.set_title(f'Bin {binno+1}\n'+fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        
        # Legend only in first subplot
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)

    fig.suptitle(r"cloud-mean accum. $\overline{M}$ (tonnes)", fontsize=24)
    plt.tight_layout()
    plt.show()

    return

In [ ]:
compare_mf(sum_mfs, mean_mfs, casenames, casecolors, minmf, maxmf, nbins=15)